In [ ]:
# ============================================================
# GRAPH VISUALIZATION: FULL GRAPH + IMPORTANT SUBGRAPH
# ============================================================

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import textwrap
import community.community_louvain as community_louvain
# ============================================================
# 1. SETTINGS
# ============================================================

NODES_FILE = "nodes_final.csv"
EDGES_FILE = "edges_final.csv"

SHOW_EDGE_LABELS = True

# Number of most important nodes to select automatically
TOP_N_IMPORTANT_NODES = 12

# Additional important nodes to always show
MANUAL_IMPORTANT_NODES = [
    "Jeffrey Epstein",
    "Palm Beach Police Department",
    "U.S. Attorney's Office for the Southern District of Florida",
    "Harvard University",
    "MC2 modeling agency",
    "Alan M. Dershowitz",
    "FBI",
    "Jean-Luc Didier Henri-Rene Brunel",
    "Bradley James Edwards",
    "Haley Robson"
]

# Node sizes
FULL_GRAPH_NODE_MULTIPLIER = 1.8
IMPORTANT_GRAPH_NODE_MULTIPLIER = 2.4

# Edge thickness
FULL_GRAPH_EDGE_MULTIPLIER = 0.35
IMPORTANT_GRAPH_EDGE_MULTIPLIER = 0.65

# Layout compactness
FULL_GRAPH_K = 0.45
IMPORTANT_GRAPH_K = 0.65

# Figures
FULL_GRAPH_FIGSIZE = (24, 14)
IMPORTANT_GRAPH_FIGSIZE = (22, 13)


# ============================================================
# 2. READ DATA
# ============================================================

nodes_df = pd.read_csv(NODES_FILE)
edges_df = pd.read_csv(EDGES_FILE)

# Ensure required columns exist
required_node_cols = ["Id", "Label"]
required_edge_cols = ["Source", "Target"]

for col in required_node_cols:
    if col not in nodes_df.columns:
        raise ValueError(f"Missing column in nodes.csv: {col}")

for col in required_edge_cols:
    if col not in edges_df.columns:
        raise ValueError(f"Missing column in edges.csv: {col}")

# Fill optional columns
if "Weight" not in edges_df.columns:
    edges_df["Weight"] = 1

if "Label" not in edges_df.columns:
    edges_df["Label"] = ""

if "action_class" not in edges_df.columns:
    edges_df["action_class"] = ""

if "node_type" not in nodes_df.columns:
    nodes_df["node_type"] = ""

if "entity_type" not in nodes_df.columns:
    nodes_df["entity_type"] = ""

if "hop_distance" not in nodes_df.columns:
    nodes_df["hop_distance"] = ""


# ============================================================
# 3. BUILD GRAPH
# ============================================================

G = nx.Graph()

# Add nodes
for _, row in nodes_df.iterrows():
    node_id = str(row["Id"])

    G.add_node(
        node_id,
        label=str(row.get("Label", node_id)),
        node_type=str(row.get("node_type", "")),
        entity_type=str(row.get("entity_type", "")),
        hop_distance=row.get("hop_distance", "")
    )

# Add edges
for _, row in edges_df.iterrows():
    source = str(row["Source"])
    target = str(row["Target"])

    if source not in G:
        G.add_node(source, label=source, node_type="", entity_type="", hop_distance="")

    if target not in G:
        G.add_node(target, label=target, node_type="", entity_type="", hop_distance="")

    weight = row.get("Weight", 1)

    try:
        weight = float(weight)
    except:
        weight = 1

    G.add_edge(
        source,
        target,
        weight=weight,
        label=str(row.get("Label", "")),
        action_class=str(row.get("action_class", "")),
        sample_claim_ids=str(row.get("sample_claim_ids", "")),
        sample_doc_ids=str(row.get("sample_doc_ids", ""))
    )


# ============================================================
# 4. CALCULATE NETWORK METRICS
# ============================================================

degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G, weight="weight", normalized=True)
closeness_centrality = nx.closeness_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G, weight="weight", max_iter=1000)

weighted_degree = dict(G.degree(weight="weight"))

nx.set_node_attributes(G, degree_centrality, "degree_centrality")
nx.set_node_attributes(G, betweenness_centrality, "betweenness_centrality")
nx.set_node_attributes(G, closeness_centrality, "closeness_centrality")
nx.set_node_attributes(G, eigenvector_centrality, "eigenvector_centrality")
nx.set_node_attributes(G, weighted_degree, "weighted_degree")

density = nx.density(G)

if nx.is_connected(G):
    average_shortest_path = nx.average_shortest_path_length(G)
else:
    largest_component = max(nx.connected_components(G), key=len)
    G_largest = G.subgraph(largest_component)
    average_shortest_path = nx.average_shortest_path_length(G_largest)

# Louvain modularity
partition = community_louvain.best_partition(G, weight="weight")
modularity = community_louvain.modularity(partition, G, weight="weight")
nx.set_node_attributes(G, partition, "community")

print("====================================================")
print("GENERAL NETWORK STATISTICS")
print("====================================================")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Density: {density:.4f}")
print(f"Modularity: {modularity:.4f}")
print(f"Average shortest path: {average_shortest_path:.4f}")
print(f"Number of communities: {len(set(partition.values()))}")
print("")


# ============================================================
# 5. SELECT IMPORTANT NODES
# ============================================================

importance_df = pd.DataFrame({
    "node": list(G.nodes()),
    "weighted_degree": [weighted_degree.get(n, 0) for n in G.nodes()],
    "degree_centrality": [degree_centrality.get(n, 0) for n in G.nodes()],
    "betweenness_centrality": [betweenness_centrality.get(n, 0) for n in G.nodes()],
    "eigenvector_centrality": [eigenvector_centrality.get(n, 0) for n in G.nodes()]
})

# Combined importance score
importance_df["importance_score"] = (
    importance_df["weighted_degree"].rank(pct=True) * 0.40 +
    importance_df["degree_centrality"].rank(pct=True) * 0.20 +
    importance_df["betweenness_centrality"].rank(pct=True) * 0.25 +
    importance_df["eigenvector_centrality"].rank(pct=True) * 0.15
)

importance_df = importance_df.sort_values("importance_score", ascending=False)

top_auto_nodes = importance_df.head(TOP_N_IMPORTANT_NODES)["node"].tolist()

important_nodes = set(top_auto_nodes)

for node in MANUAL_IMPORTANT_NODES:
    if node in G:
        important_nodes.add(node)

print("====================================================")
print("MOST IMPORTANT NODES")
print("====================================================")
print(
    importance_df[
        ["node", "weighted_degree", "degree_centrality", "betweenness_centrality", "eigenvector_centrality", "importance_score"]
    ].head(20).to_string(index=False)
)
print("")


# ============================================================
# 6. SUBGRAPH WITH IMPORTANT NODES + THEIR CONNECTIONS
# ============================================================

subgraph_nodes = set(important_nodes)

# Add all direct neighbors of the important nodes
for node in important_nodes:
    if node in G:
        subgraph_nodes.update(G.neighbors(node))

G_important = G.subgraph(subgraph_nodes).copy()

print("====================================================")
print("SUBGRAPH STATISTICS")
print("====================================================")
print(f"Number of nodes in important subgraph: {G_important.number_of_nodes()}")
print(f"Number of edges in important subgraph: {G_important.number_of_edges()}")
print(f"Density important subgraph: {nx.density(G_important):.4f}")
print("")


# ============================================================
# 7. VISUAL HELPER FUNCTIONS
# ============================================================

def shorten_label(label, max_len=24):
    label = str(label)
    if len(label) <= max_len:
        return label
    return "\n".join(textwrap.wrap(label, max_len))


def get_node_color(node, graph):
    node_type = graph.nodes[node].get("node_type", "")
    entity_type = graph.nodes[node].get("entity_type", "")

    if node_type == "core_organization":
        return "orange"

    if entity_type in ["victim", "minor"]:
        return "lightcoral"

    if entity_type in ["law enforcement", "law_enforcement"]:
        return "lightskyblue"

    if entity_type in ["legal counsel", "legal_counsel", "lawyer"]:
        return "lightgreen"

    if entity_type in ["business associate", "business_associate", "trafficker_accomplice"]:
        return "plum"

    if entity_type in ["government_official", "government official"]:
        return "khaki"

    return "lightgray"


def get_edge_color(edge_data):
    action_class = edge_data.get("action_class", "")

    if action_class == "investigative":
        return "#4C78A8"

    if action_class == "legal_procedural":
        return "#F58518"

    if action_class == "academic_philanthropic":
        return "#54A24B"

    if action_class == "business_recruitment":
        return "#B279A2"

    return "#BBBBBB"


def calculate_node_sizes(graph, multiplier=1.8):
    sizes = []

    for node, data in graph.nodes(data=True):
        node_type = data.get("node_type", "")
        w_degree = data.get("weighted_degree", 1)

        try:
            w_degree = float(w_degree)
        except:
            w_degree = 1

        if node_type == "core_organization":
            size = 2600
        else:
            size = 700 + w_degree * 55

        size = size * multiplier
        size = min(size, 8500)

        sizes.append(size)

    return sizes


def calculate_edge_widths(graph, multiplier=0.45):
    widths = []

    for _, _, data in graph.edges(data=True):
        weight = data.get("weight", 1)

        try:
            weight = float(weight)
        except:
            weight = 1

        width = 1.4 + weight * multiplier
        width = min(width, 10)
        widths.append(width)

    return widths


def draw_graph(graph, title, figsize, k_value, node_multiplier, edge_multiplier, filename):
    plt.figure(figsize=figsize)

    # Layout
    pos = nx.spring_layout(
        graph,
        seed=42,
        k=k_value,
        iterations=400,
        weight="weight"
    )

    node_colors = [get_node_color(node, graph) for node in graph.nodes()]
    node_sizes = calculate_node_sizes(graph, multiplier=node_multiplier)
    edge_colors = [get_edge_color(data) for _, _, data in graph.edges(data=True)]
    edge_widths = calculate_edge_widths(graph, multiplier=edge_multiplier)

    # Edges
    nx.draw_networkx_edges(
        graph,
        pos,
        edge_color=edge_colors,
        width=edge_widths,
        alpha=0.55
    )

    # Nodes
    nx.draw_networkx_nodes(
        graph,
        pos,
        node_color=node_colors,
        node_size=node_sizes,
        edgecolors="black",
        linewidths=0.8,
        alpha=0.95
    )

    # Node labels
    node_labels = {
        node: shorten_label(graph.nodes[node].get("label", node), max_len=26)
        for node in graph.nodes()
    }

    nx.draw_networkx_labels(
        graph,
        pos,
        labels=node_labels,
        font_size=12,
        font_weight="bold"
    )

    # Edge labels
    if SHOW_EDGE_LABELS:
        edge_labels = {}

        for u, v, data in graph.edges(data=True):
            label = data.get("label", "")

            if label and label != "nan":
                edge_labels[(u, v)] = shorten_label(label, max_len=28)

        nx.draw_networkx_edge_labels(
            graph,
            pos,
            edge_labels=edge_labels,
            font_size=9,
            font_color="darkred",
            rotate=False,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.65)
        )

    # Legend
    legend_elements = [
        Patch(facecolor="orange", edgecolor="black", label="Core organization"),
        Patch(facecolor="lightcoral", edgecolor="black", label="Victim / minor"),
        Patch(facecolor="lightskyblue", edgecolor="black", label="Law enforcement"),
        Patch(facecolor="lightgreen", edgecolor="black", label="Legal counsel / lawyer"),
        Patch(facecolor="plum", edgecolor="black", label="Business / recruitment"),
        Patch(facecolor="khaki", edgecolor="black", label="Government official"),
        Patch(facecolor="lightgray", edgecolor="black", label="Other")
    ]

    plt.legend(
        handles=legend_elements,
        loc="upper left",
        fontsize=10,
        frameon=True
    )

    plt.title(title, fontsize=18, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()

    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Graph saved as: {filename}")


# ============================================================
# 8. GRAPH 1: FULL GRAPH
# ============================================================

draw_graph(
    graph=G,
    title="Full network graph",
    figsize=FULL_GRAPH_FIGSIZE,
    k_value=FULL_GRAPH_K,
    node_multiplier=FULL_GRAPH_NODE_MULTIPLIER,
    edge_multiplier=FULL_GRAPH_EDGE_MULTIPLIER,
    filename="full_graph.png"
)


# ============================================================
# 9. EXTRA: EXPORT TABLES
# ============================================================

importance_df.to_csv("node_importance_scores.csv", index=False)

subgraph_edges_df = nx.to_pandas_edgelist(G_important)
subgraph_edges_df.to_csv("important_subgraph_edges.csv", index=False)

subgraph_nodes_data = []

for node, data in G_important.nodes(data=True):
    row = {
        "Id": node,
        "Label": data.get("label", node),
        "node_type": data.get("node_type", ""),
        "entity_type": data.get("entity_type", ""),
        "weighted_degree": data.get("weighted_degree", 0),
        "degree_centrality": data.get("degree_centrality", 0),
        "betweenness_centrality": data.get("betweenness_centrality", 0),
        "closeness_centrality": data.get("closeness_centrality", 0),
        "eigenvector_centrality": data.get("eigenvector_centrality", 0),
        "community": data.get("community", "")
    }
    subgraph_nodes_data.append(row)

subgraph_nodes_df = pd.DataFrame(subgraph_nodes_data)
subgraph_nodes_df.to_csv("important_subgraph_nodes.csv", index=False)

print("")
print("====================================================")
print("FILES CREATED")
print("====================================================")
print("1. full_graph.png")
print("2. important_nodes_graph.png")
print("3. node_importance_scores.csv")
print("4. important_subgraph_nodes.csv")
print("5. important_subgraph_edges.csv")